# Convert raw data to 'strict' json

In [5]:
import json
import gzip
import os
dataset_name = "Beauty"
os.makedirs(dataset_name, exist_ok=True)

def parse(path):
  g = gzip.open(path, 'r')
  for l in g:
    yield json.dumps(eval(l))

# Beauty dataset
f = open(f"./{dataset_name}/{dataset_name}.json", 'w')
for l in parse(f"reviews_{dataset_name}_5.json.gz"):
  f.write(l + '\n')

In [6]:
# print the number of lines in the file and the first line
data = open(f"./{dataset_name}/{dataset_name}.json", 'r')
print("Number of lines:", sum(1 for _ in data))
data.seek(0)  # Reset file pointer to the beginning
print("First line:", data.readline().strip())
data.close()

Number of lines: 198502
First line: {"reviewerID": "A1YJEY40YUW4SE", "asin": "7806397051", "reviewerName": "Andrea", "helpful": [3, 4], "reviewText": "Very oily and creamy. Not at all what I expected... ordered this to try to highlight and contour and it just looked awful!!! Plus, took FOREVER to arrive.", "overall": 1.0, "summary": "Don't waste your money", "unixReviewTime": 1391040000, "reviewTime": "01 30, 2014"}


In [3]:
# 👈👈👈
import numpy as np
import pandas as pd
import json

# Initialize mapping dictionaries
userID_mapping = {}
itemID_mapping = {}

# Open the JSON file for reading
data = open(f"./{dataset_name}/{dataset_name}.json", 'r')

# Initialize lists to store userID, itemID, and timestamp
userIDs = []
itemIDs = []
timestamps = []

# Process each line in the JSON file
for line in data:
    review = json.loads(line.strip())
    userID = review['reviewerID']
    itemID = review['asin']
    timestamp = review['unixReviewTime']
    
    # Map userID to an integer starting from 1
    if userID not in userID_mapping:
        userID_mapping[userID] = len(userID_mapping) + 1
    
    # Map itemID to an integer starting from 1
    if itemID not in itemID_mapping:
        itemID_mapping[itemID] = len(itemID_mapping) + 1
    
    # Append mapped values and timestamp to lists
    userIDs.append(userID_mapping[userID])
    itemIDs.append(itemID_mapping[itemID])
    timestamps.append(timestamp)

# Save mapping dictionaries as .npy files
np.save(f'./{dataset_name}/user_mapping.npy', userID_mapping)
print("user_num:", len(userID_mapping))
print("the first five userID mapping:", list(userID_mapping.items())[:5])
np.save(f'./{dataset_name}/item_mapping.npy', itemID_mapping)
print("item_num:", len(itemID_mapping))
print("the first five itemID mapping:", list(itemID_mapping.items())[:5])

# Group itemIDs by userID and sort by timestamp
user_item_mapping = {}
for userID, itemID, timestamp in zip(userIDs, itemIDs, timestamps):
    if userID not in user_item_mapping:
        user_item_mapping[userID] = []
    user_item_mapping[userID].append((itemID, timestamp))

# Sort itemIDs for each user by timestamp
for userID in user_item_mapping:
    user_item_mapping[userID].sort(key=lambda x: x[1])
    user_item_mapping[userID] = [item[0] for item in user_item_mapping[userID]]

# Print a sample of the results
print("user-item mapping:", list(user_item_mapping.items())[:5])

# Split data into training, validation, and testing sets using leave-one-out strategy
train_data = {}
val_data = {}
test_data = {}

for userID, item_sequence in user_item_mapping.items():
    # Assign the last item for testing, the second-to-last for validation, and the rest for training
    train_data[userID] = item_sequence[:-2]
    val_data[userID] = item_sequence[:-1]
    test_data[userID] = item_sequence

# Print a sample of the split data
# print("training data:", list(train_data.items())[:5])
# print("validation data:", list(val_data.items())[:5])
# print("testing data:", list(test_data.items())[:5])

# Prepare data for train, validation, and test sets
def prepare_data(data_dict):
    rows = []
    for userID, item_sequence in data_dict.items():
        history = item_sequence[:-1]
        target = item_sequence[-1]
        rows.append({'user': userID, 'history': history, 'target': target})
    return pd.DataFrame(rows)

# Create dataframes for train, validation, and test sets
train_df = prepare_data(train_data)
print("\nTraining data shape:", train_df.shape)
print("the first 3 rows of training data:\n", train_df.head(3))
val_df = prepare_data(val_data)
print("\nValidation data shape:", val_df.shape)
print("the first 3 rows of validation data:\n", val_df.head(3))
test_df = prepare_data(test_data)
print("\nTesting data shape:", test_df.shape)
print("the first 3 rows of testing data:\n", test_df.head(3))

# Save dataframes to parquet files
train_df.to_parquet(f'./{dataset_name}/train.parquet', index=False)
val_df.to_parquet(f'./{dataset_name}/valid.parquet', index=False)
test_df.to_parquet(f'./{dataset_name}/test.parquet', index=False)

print("Data saved to parquet files.")

data.close()


user_num: 22363
the first five userID mapping: [('A1YJEY40YUW4SE', 1), ('A60XNB876KYML', 2), ('A3G6XNM240RMWA', 3), ('A1PQFP6SAJ6D80', 4), ('A38FVHZTNQ271F', 5)]
item_num: 12101
the first five itemID mapping: [('7806397051', 1), ('9759091062', 2), ('9788072216', 3), ('9790790961', 4), ('9790794231', 5)]
user-item mapping: [(1, [6846, 7873, 4585, 1, 5406]), (2, [816, 10406, 11194, 11651, 9716, 1, 233]), (3, [1, 6050, 7977, 5252, 4211, 243, 11204, 5863, 6609]), (4, [5522, 439, 5161, 11140, 1, 7849]), (5, [1, 10470, 10064, 9403, 10362, 4758, 6500, 11444, 11390])]

Training data shape: (22363, 3)
the first 3 rows of training data:
    user                           history  target
0     1                      [6846, 7873]    4585
1     2        [816, 10406, 11194, 11651]    9716
2     3  [1, 6050, 7977, 5252, 4211, 243]   11204

Validation data shape: (22363, 3)
the first 3 rows of validation data:
    user                                  history  target
0     1                       [684

# Generate Item Semantic Embeddings

In [8]:
# # Beauty metadata
# f = open(f"./{dataset_name}/{dataset_name}_metadata.json", 'w')
# for l in parse(f"meta_{dataset_name}.json.gz"):
#   f.write(l + '\n')

In [9]:
# # Open the metadata file for reading
# with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
#     # Create a reverse mapping from itemID to asin
#     reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}
#
#     # Initialize a dictionary to store the extracted information
#     item_info = {}
#
#     # Process each line in the metadata file
#     for line in metadata_file:
#         metadata = json.loads(line.strip())
#         asin = metadata.get('asin')
#
#         # Check if the asin exists in the reverse mapping
#         if asin in reverse_itemID_mapping.values():
#             itemID = itemID_mapping[asin]
#             item_info[itemID] = {
#                 'title': metadata.get('title') if metadata.get('title') else None,
#                 'price': metadata.get('price') if metadata.get('price') else None,
#                 'salesRank': metadata.get('salesRank') if metadata.get('salesRank') else None,
#                 'brand': metadata.get('brand') if metadata.get('brand') else None,
#                 'categories': metadata.get('categories') if metadata.get('categories') else None,
#             }
#
# # Print the information for the first 5 items
# for itemID, info in list(item_info.items())[:5]:
#     print(f"ItemID: {itemID}, Info: {info}")

ItemID: 1, Info: {'title': 'WAWO 15 Color Professionl Makeup Eyeshadow Camouflage Facial Concealer Neutral Palette', 'price': 5.04, 'salesRank': {'Beauty': 10486}, 'brand': 'COKA', 'categories': [['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers']]}
ItemID: 2, Info: {'title': 'Xtreme Brite Brightening Gel 1oz.', 'price': 19.99, 'salesRank': {'Beauty': 52254}, 'brand': 'Xtreme Brite', 'categories': [['Beauty', 'Hair Care', 'Styling Products', 'Creams, Gels & Lotions']]}
ItemID: 3, Info: {'title': 'Prada Candy By Prada Eau De Parfum Spray 1.7 Oz For Women', 'price': 65.86, 'salesRank': {'Beauty': 78916}, 'brand': 'Prada', 'categories': [['Beauty', 'Fragrance', "Women's", 'Eau de Parfum']]}
ItemID: 4, Info: {'title': 'Versace Bright Crystal Eau de Toilette Spray for Women, 3 Ounce', 'price': 52.33, 'salesRank': {'Beauty': 764}, 'brand': 'Versace', 'categories': [['Beauty', 'Fragrance', "Women's", 'Eau de Toilette']]}
ItemID: 5, Info: {'title': 'Stella McCartney Stella', 'price': Non

In [3]:
# # 👈👈👈
# dataset_name = "Beauty"
# # Open the metadata file for reading
#
# set_c = set()
# set_r = set()
# with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
#     # Create a reverse mapping from itemID to asin
#     reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}
#
#     # Process each line in the metadata file
#     for line in metadata_file:
#         metadata = json.loads(line.strip())
#         asin = metadata.get('asin')
#
#         # Check if the asin exists in the reverse mapping
#         if asin in reverse_itemID_mapping.values():
#             c = metadata.get('categories')[0] if metadata.get('categories') else None
#             for cc in c:
#                 set_c.add(cc)
#
#             r = metadata.get('salesRank') if metadata.get('salesRank') else None
#             if r:
#                 for rr in r:
#                     set_r.add(rr)
#
# print(set_c)
# print(set_r)

KeyboardInterrupt: 

In [5]:
# 👈👈👈
dataset_name = "Beauty"
# Open the metadata file for reading

with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
    # Create a reverse mapping from itemID to asin
    reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}

    # Initialize a dictionary to store the extracted information
    item_info = {}

    # Process each line in the metadata file
    for line in metadata_file:
        metadata = json.loads(line.strip())
        asin = metadata.get('asin')

        # Check if the asin exists in the reverse mapping
        if asin in reverse_itemID_mapping.values():
            itemID = itemID_mapping[asin]
            item_info[itemID] = {
                # 1. text
                # 'xxxxxxx'
                'title': metadata.get('title') if metadata.get('title') else None,
                # 190
                'price': metadata.get('price') if metadata.get('price') else None,
                # {'Beauty': 10486, "xxx": xxx}
                'salesRank': metadata.get('salesRank') if metadata.get('salesRank') else None,
                # 'COKA'
                'brand': metadata.get('brand') if metadata.get('brand') else None,
                # ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers']
                'categories': metadata.get('categories')[0] if metadata.get('categories') else None,
                # # 'xxxxxxxxxxxxxxxx'
                # 'description': metadata.get('description') if metadata.get('description') else None,

                # 2. image
                # 'imUrl': 'http://ecx.images-amazon.com/images/I/41Rn18OeU6L._SY300_.jpg'
                'imUrl': metadata.get('imUrl') if metadata.get('imUrl') else None,
            }
        # asin = metadata.get('asin')
        #
        # # Check if the asin exists in the reverse mapping
        # if asin in reverse_itemID_mapping.values():
        #     # for k in metadata.keys():
        #     #     print(k, ": ", metadata[k])
        #     # break

# Print the information for the first 5 items
for itemID, info in list(item_info.items())[:5]:
    print(f"ItemID: {itemID}, Info: {info}")

ItemID: 1, Info: {'title': 'WAWO 15 Color Professionl Makeup Eyeshadow Camouflage Facial Concealer Neutral Palette', 'price': 5.04, 'salesRank': {'Beauty': 10486}, 'brand': 'COKA', 'categories': ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers'], 'imUrl': 'http://ecx.images-amazon.com/images/I/41Rn18OeU6L._SY300_.jpg'}
ItemID: 2, Info: {'title': 'Xtreme Brite Brightening Gel 1oz.', 'price': 19.99, 'salesRank': {'Beauty': 52254}, 'brand': 'Xtreme Brite', 'categories': ['Beauty', 'Hair Care', 'Styling Products', 'Creams, Gels & Lotions'], 'imUrl': 'http://ecx.images-amazon.com/images/I/41QWW9v18XL._SY300_.jpg'}
ItemID: 3, Info: {'title': 'Prada Candy By Prada Eau De Parfum Spray 1.7 Oz For Women', 'price': 65.86, 'salesRank': {'Beauty': 78916}, 'brand': 'Prada', 'categories': ['Beauty', 'Fragrance', "Women's", 'Eau de Parfum'], 'imUrl': 'http://ecx.images-amazon.com/images/I/51iT2k6LPYL._SY300_.jpg'}
ItemID: 4, Info: {'title': 'Versace Bright Crystal Eau de Toilette Spray for Wome

In [1]:
# # 👈👈👈
# dataset_name = "Beauty"
# # Open the metadata file for reading
#
# with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
#     # Create a reverse mapping from itemID to asin
#     reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}
#
#     # Initialize a dictionary to store the extracted information
#     item_info = {}
#
#     # Process each line in the metadata file
#     for line in metadata_file:
#         metadata = json.loads(line.strip())
#         asin = metadata.get('asin')
#
#         # Check if the asin exists in the reverse mapping
#         if asin in reverse_itemID_mapping.values():
#             itemID = itemID_mapping[asin]
#             item_info[itemID] = {
#                 # 1. text
#                 # 'xxxxxxx'
#                 'title': metadata.get('title') if metadata.get('title') else None,
#                 # 190
#                 'price': metadata.get('price') if metadata.get('price') else None,
#                 # {'Beauty': 10486, "xxx": xxx}
#                 'salesRank': metadata.get('salesRank') if metadata.get('salesRank') else None,
#                 # 'COKA'
#                 'brand': metadata.get('brand') if metadata.get('brand') else None,
#                 # ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers']
#                 'categories': metadata.get('categories')[0] if metadata.get('categories') else None,
#                 # # 'xxxxxxxxxxxxxxxx'
#                 # 'description': metadata.get('description') if metadata.get('description') else None,
#
#                 # 2. image
#                 # 'imUrl': 'http://ecx.images-amazon.com/images/I/41Rn18OeU6L._SY300_.jpg'
#                 'imUrl': metadata.get('imUrl') if metadata.get('imUrl') else None,
#             }
#         # asin = metadata.get('asin')
#         #
#         # # Check if the asin exists in the reverse mapping
#         # if asin in reverse_itemID_mapping.values():
#         #     # for k in metadata.keys():
#         #     #     print(k, ": ", metadata[k])
#         #     # break
#
# # Print the information for the first 5 items
# for itemID, info in list(item_info.items())[:5]:
#     print(f"ItemID: {itemID}, Info: {info}")

NameError: name 'itemID_mapping' is not defined

In [1]:
# # 👈👈👈
# dataset_name = "Beauty"
# # Open the metadata file for reading
#
# with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
#     # Create a reverse mapping from itemID to asin
#     reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}
#
#     # Initialize a dictionary to store the extracted information
#     item_info = {}
#
#     # Process each line in the metadata file
#     for line in metadata_file:
#         metadata = json.loads(line.strip())
#         asin = metadata.get('asin')
#
#         # Check if the asin exists in the reverse mapping
#         if asin in reverse_itemID_mapping.values():
#             itemID = itemID_mapping[asin]
#             item_info[itemID] = {
#                 # 1. text
#                 # 'xxxxxxx'
#                 'title': metadata.get('title') if metadata.get('title') else None,
#                 # 190
#                 'price': metadata.get('price') if metadata.get('price') else None,
#                 # {'Beauty': 10486, "xxx": xxx}
#                 'salesRank': metadata.get('salesRank') if metadata.get('salesRank') else None,
#                 # 'COKA'
#                 'brand': metadata.get('brand') if metadata.get('brand') else None,
#                 # ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers']
#                 'categories': metadata.get('categories')[0] if metadata.get('categories') else None,
#                 # # 'xxxxxxxxxxxxxxxx'
#                 # 'description': metadata.get('description') if metadata.get('description') else None,
#
#                 # 2. image
#                 # 'imUrl': 'http://ecx.images-amazon.com/images/I/41Rn18OeU6L._SY300_.jpg'
#                 'imUrl': metadata.get('imUrl') if metadata.get('imUrl') else None,
#             }
#         # asin = metadata.get('asin')
#         #
#         # # Check if the asin exists in the reverse mapping
#         # if asin in reverse_itemID_mapping.values():
#         #     # for k in metadata.keys():
#         #     #     print(k, ": ", metadata[k])
#         #     # break
#
# # Print the information for the first 5 items
# for itemID, info in list(item_info.items())[:5]:
#     print(f"ItemID: {itemID}, Info: {info}")

NameError: name 'itemID_mapping' is not defined

In [1]:
# # 👈👈👈
# dataset_name = "Beauty"
# # Open the metadata file for reading
#
# with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
#     # Create a reverse mapping from itemID to asin
#     reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}
#
#     # Initialize a dictionary to store the extracted information
#     item_info = {}
#
#     # Process each line in the metadata file
#     for line in metadata_file:
#         metadata = json.loads(line.strip())
#         asin = metadata.get('asin')
#
#         # Check if the asin exists in the reverse mapping
#         if asin in reverse_itemID_mapping.values():
#             itemID = itemID_mapping[asin]
#             item_info[itemID] = {
#                 # 1. text
#                 # 'xxxxxxx'
#                 'title': metadata.get('title') if metadata.get('title') else None,
#                 # 190
#                 'price': metadata.get('price') if metadata.get('price') else None,
#                 # {'Beauty': 10486, "xxx": xxx}
#                 'salesRank': metadata.get('salesRank') if metadata.get('salesRank') else None,
#                 # 'COKA'
#                 'brand': metadata.get('brand') if metadata.get('brand') else None,
#                 # ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers']
#                 'categories': metadata.get('categories')[0] if metadata.get('categories') else None,
#                 # # 'xxxxxxxxxxxxxxxx'
#                 # 'description': metadata.get('description') if metadata.get('description') else None,
#
#                 # 2. image
#                 # 'imUrl': 'http://ecx.images-amazon.com/images/I/41Rn18OeU6L._SY300_.jpg'
#                 'imUrl': metadata.get('imUrl') if metadata.get('imUrl') else None,
#             }
#         # asin = metadata.get('asin')
#         #
#         # # Check if the asin exists in the reverse mapping
#         # if asin in reverse_itemID_mapping.values():
#         #     # for k in metadata.keys():
#         #     #     print(k, ": ", metadata[k])
#         #     # break
#
# # Print the information for the first 5 items
# for itemID, info in list(item_info.items())[:5]:
#     print(f"ItemID: {itemID}, Info: {info}")

NameError: name 'itemID_mapping' is not defined

In [6]:
# Prepare data for embedding
item_embeddings = []
for itemID, info in item_info.items():
    # Combine relevant fields into a single text for embedding

    item_embeddings.append({
        'ItemID': itemID,
        'title': info.get('title', ''),
        'price': info.get('price', ''),
        'salesRank': info.get('salesRank', ''),
        'brand': info.get('brand', ''),
        'categories': info.get('categories', ''),
        'image': info.get('imUrl', ''),
    })

# Convert to DataFrame
item_emb_df = pd.DataFrame(item_embeddings)

print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))


Item embeddings DataFrame shape: (12100, 7)
The first 3 rows of item embeddings DataFrame:
    ItemID                                              title  price  \
0       1  WAWO 15 Color Professionl Makeup Eyeshadow Cam...   5.04   
1       2                  Xtreme Brite Brightening Gel 1oz.  19.99   
2       3  Prada Candy By Prada Eau De Parfum Spray 1.7 O...  65.86   

           salesRank         brand  \
0  {'Beauty': 10486}          COKA   
1  {'Beauty': 52254}  Xtreme Brite   
2  {'Beauty': 78916}         Prada   

                                          categories  \
0  [Beauty, Makeup, Face, Concealers & Neutralizers]   
1  [Beauty, Hair Care, Styling Products, Creams, ...   
2        [Beauty, Fragrance, Women's, Eau de Parfum]   

                                               image  
0  http://ecx.images-amazon.com/images/I/41Rn18Oe...  
1  http://ecx.images-amazon.com/images/I/41QWW9v1...  
2  http://ecx.images-amazon.com/images/I/51iT2k6L...  


In [7]:
import pandas as pd
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm
import numpy as np
import os
import warnings

# ====== 额外引入你的 build_limited_text 函数 ======
from transformers import CLIPProcessor
import transformers

# ====== 🔇 静音 transformers 输出 ======
warnings.filterwarnings("ignore", message="Token indices sequence length is longer*")
transformers.logging.set_verbosity_error()

clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
tokenizer = clip_processor.tokenizer

TITLE_LIMIT = 25
CLIP_LIMIT = 77
RESERVED = 2
SAFE_LIMIT = CLIP_LIMIT - RESERVED

def build_limited_text(info):
    """拼接文本：
      - title ≤ 25 tokens
      - 整体 ≤ 77 tokens (含 [CLS], [EOS])
    """
    raw_title = str(info.get("title", ""))
    title_tokens = tokenizer.encode(raw_title, add_special_tokens=False)[:TITLE_LIMIT]
    title_text = tokenizer.decode(title_tokens)
    text = f"'title': {title_text}"
    current_len = len(tokenizer.encode(text, add_special_tokens=False))

    fields = [
        ("categories", info.get("categories", "")),
        ("price", info.get("price", "")),
        ("salesRank", info.get("salesRank", "")),
        ("brand", info.get("brand", "")),
    ]

    for k, v in fields:
        segment = f"\n'{k}': {v}"
        tokens = tokenizer.encode(text + segment, add_special_tokens=False)
        if len(tokens) <= SAFE_LIMIT:
            text += segment
            current_len = len(tokens)
        else:
            allowed = SAFE_LIMIT - current_len
            if allowed > 0:
                extra_tokens = tokenizer.encode(segment, add_special_tokens=False)
                text += tokenizer.decode(extra_tokens[:allowed])
            break
    return text.strip()
# ====================================================


# ========== 1️⃣ 基础设置 ==========
dataset_name = "Beauty"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

# ========== 2️⃣ 辅助函数 ==========
def get_image_emb(url):
    """从URL获取图像embedding，失败则返回全0向量"""
    if not url or not isinstance(url, str) or not url.startswith("http"):
        return np.zeros(512)
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        inputs = clip_processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb.squeeze().cpu().numpy()
    except Exception:
        return np.zeros(512)


def get_text_emb(info):
    """获取文本embedding，使用 build_limited_text 限制 token 数"""
    text = build_limited_text(info)
    if not text or not isinstance(text, str):
        return np.zeros(512)
    inputs = clip_processor(text=[text], return_tensors="pt",
                            padding=True, truncation=True, max_length=77).to(device)

    # ✅ 打印 token 序列长度
    input_len = inputs["input_ids"].shape[-1]
    if input_len > 77:
        print(f"[WARN] 超过长度: {input_len}")

    with torch.no_grad():
        emb = clip_model.get_text_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu().numpy()


# ========== 3️⃣ 生成embedding ==========
text_embs, img_embs = [], []

print(f"Encoding {len(item_emb_df)} items with CLIP (title≤25, total≤77)...")

for _, row in tqdm(item_emb_df.iterrows(), total=len(item_emb_df), desc="CLIP encoding"):
    text_emb = get_text_emb(row).tolist()  # 🚨 改：传入整行 row（含 title/price/...）
    img_emb = get_image_emb(row["image"]).tolist()
    text_embs.append(text_emb)
    img_embs.append(img_emb)

# ========== 4️⃣ 保存 ==========
item_emb_df_ = item_emb_df.copy()
item_emb_df_["text_emb"] = text_embs
item_emb_df_["image_emb"] = img_embs

print("✅ Embedding generation completed.")

# ========== 3️⃣ 生成embedding ==========
# text_embs, img_embs = [], []
#
# print(f"Encoding {len(item_info)} items with CLIP (title≤25, total≤77)...")
# item_embeddings = []
# for itemID, info in tqdm(item_info.items(), total=len(item_info), desc="CLIP encoding"):
#     text_emb = get_text_emb(info).tolist()  # 🚨 改：传入整行 row（含 title/price/...）
#     img_emb = get_image_emb(info["imUrl"]).tolist()
#
#     text_embs.append(text_emb)
#     img_embs.append(img_emb)
#
#     item_embeddings.append({'ItemID': itemID, 'text_emb': text_embs, 'img_emb': img_embs})
#
# # ========== 4️⃣ 保存 ==========
# item_emb_df = pd.DataFrame(item_embeddings)
#
# print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
# print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))

D:\2025南航实习\GlobalPointer_pytorch-main\.venv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Encoding 12100 items with CLIP (title≤25, total≤77)...


CLIP encoding: 100%|██████████| 12100/12100 [14:54<00:00, 13.53it/s]

✅ Embedding generation completed.


In [8]:
item_emb_df_.to_parquet(f'./{dataset_name}/item_mul_emb1.parquet', index=False)

In [10]:
item_emb_df_2 = item_emb_df_.copy()
item_emb_df_2["embedding"] = item_emb_df_2.apply(
    lambda row: np.concatenate([np.array(row["text_emb"]), np.array(row["image_emb"])]).tolist(),
    axis=1
)
# 和下面二选一

In [11]:
item_emb_df_2.head(5)

,ItemID,title,price,salesRank,brand,categories,image,text_emb,image_emb,embedding
0,1,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,5.04,{'Beauty': 10486},COKA,"[Beauty, Makeup, Face, Concealers & Neutralizers]",http://ecx.images-amazon.com/images/I/41Rn18Oe...,"[0.0023019001819193363, 0.011413156986236572, ...","[-0.023478863760828972, 0.04270932823419571, 0...","[0.0023019001819193363, 0.011413156986236572, ..."
1,2,Xtreme Brite Brightening Gel 1oz.,19.99,{'Beauty': 52254},Xtreme Brite,"[Beauty, Hair Care, Styling Products, Creams, ...",http://ecx.images-amazon.com/images/I/41QWW9v1...,"[0.015429544262588024, 0.02329571172595024, -0...","[0.01667862758040428, 0.014809303916990757, 0....","[0.015429544262588024, 0.02329571172595024, -0..."
2,3,Prada Candy By Prada Eau De Parfum Spray 1.7 O...,65.86,{'Beauty': 78916},Prada,"[Beauty, Fragrance, Women's, Eau de Parfum]",http://ecx.images-amazon.com/images/I/51iT2k6L...,"[-0.025405675172805786, -0.04978051781654358, ...","[-0.030022917315363884, -0.04415012151002884, ...","[-0.025405675172805786, -0.04978051781654358, ..."
3,4,Versace Bright Crystal Eau de Toilette Spray f...,52.33,{'Beauty': 764},Versace,"[Beauty, Fragrance, Women's, Eau de Toilette]",http://ecx.images-amazon.com/images/I/418LYGLE...,"[0.026444917544722557, -0.030652707442641258, ...","[-0.01538288313895464, -0.042142171412706375, ...","[0.026444917544722557, -0.030652707442641258, ..."
4,5,Stella McCartney Stella,NaN,{'Beauty': 142503},None,"[Beauty, Fragrance, Women's, Eau de Parfum]",http://ecx.images-amazon.com/images/I/31L2n60J...,"[0.015332376584410667, 0.0013458984903991222, ...","[-0.04838688299059868, 0.021488051861524582, 0...","[0.015332376584410667, 0.0013458984903991222, ..."


In [21]:
# item_emb_df_ = item_emb_df.copy()
# text_embs = np.stack(item_emb_df_["text_emb"].to_numpy())
# img_embs = np.stack(item_emb_df_["img_emb"].to_numpy())
#
# item_emb_df_["embedding"] = list(np.concatenate([text_embs, img_embs], axis=1))
# item_emb_df

Exception ignored in: Exception ignored in sys.unraisablehook: <built-in function unraisablehook>
Traceback (most recent call last):
  File "D:\2025南航实习\GlobalPointer_pytorch-main\.venv1\lib\site-packages\ipykernel\iostream.py", line 694, in write
    self._schedule_flush()
  File "D:\2025南航实习\GlobalPointer_pytorch-main\.venv1\lib\site-packages\ipykernel\iostream.py", line 590, in _schedule_flush
    self.pub_thread.schedule(_schedule_in_thread)
  File "D:\2025南航实习\GlobalPointer_pytorch-main\.venv1\lib\site-packages\ipykernel\iostream.py", line 267, in schedule
    self._event_pipe.send(b"")
  File "D:\2025南航实习\GlobalPointer_pytorch-main\.venv1\lib\site-packages\zmq\sugar\socket.py", line 698, in send
    return super().send(data, flags=flags, copy=copy, track=track)
  File "zmq/backend/cython/_zmq.py", line 1137, in zmq.backend.cython._zmq.Socket.send
    def send(self, data, flags=0, copy: bint = True, track: bint = False):
  File "zmq/backend/cython/_zmq.py", line 1185, in zmq.backe

KeyboardInterrupt: 

In [12]:
# Save to parquet file
item_emb_df_2.to_parquet(f'./{dataset_name}/item_mul_emb.parquet', index=False)

print("Item embeddings saved to item_emb.parquet.")

Item embeddings saved to item_emb.parquet.


In [37]:
# import pandas as pd
# import torch
# from transformers import CLIPProcessor, CLIPModel
# from PIL import Image
# import requests
# from io import BytesIO
# from tqdm import tqdm
# import numpy as np
# import os
# import warnings
#
# # 忽略不影响运行的警告
# warnings.filterwarnings("ignore", category=UserWarning)
# os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
#
# # ========== 1️⃣ 基础设置 ==========
# dataset_name = "Beauty"
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Using device:", device)
#
# clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
# clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
#
#
# # ========== 2️⃣ 定义辅助函数 ==========
# def get_image_emb(url):
#     """从URL获取图像embedding，失败则返回全0向量"""
#     if not url or not isinstance(url, str) or not url.startswith("http"):
#         return np.zeros(512)
#     try:
#         response = requests.get(url, timeout=5)
#         img = Image.open(BytesIO(response.content)).convert("RGB")
#         inputs = clip_processor(images=img, return_tensors="pt").to(device)
#         with torch.no_grad():
#             emb = clip_model.get_image_features(**inputs)
#         emb = emb / emb.norm(dim=-1, keepdim=True)
#         return emb.squeeze().cpu().numpy()
#     except Exception:
#         return np.zeros(512)
#
#
# def get_text_emb(text):
#     """获取文本embedding，自动截断到CLIP最大长度（77 tokens）"""
#     if not text or not isinstance(text, str):
#         return np.zeros(512)
#     # 截断文本以防超过CLIP最大长度
#     tokens = text.split()[:75]  # 预留 [CLS] 和 [EOS]
#     text_short = " ".join(tokens)
#     inputs = clip_processor(text=[text_short], return_tensors="pt", padding=True, truncation=True).to(device)
#     with torch.no_grad():
#         emb = clip_model.get_text_features(**inputs)
#     emb = emb / emb.norm(dim=-1, keepdim=True)
#     return emb.squeeze().cpu().numpy()
#
#
# # ========== 3️⃣ 生成embedding ==========
# text_embs, img_embs = [], []
#
# print(f"Encoding {len(item_emb_df)} items with CLIP...")
#
# for _, row in tqdm(item_emb_df.iterrows(), total=len(item_emb_df), desc="CLIP encoding"):
#     text_emb = get_text_emb(row["text"]).tolist()
#     img_emb = get_image_emb(row["image"]).tolist()
#     text_embs.append(text_emb)
#     img_embs.append(img_emb)
#
# # ========== 4️⃣ 保存 ==========
# item_emb_df_ = item_emb_df.copy()
# item_emb_df_["text_emb"] = text_embs
# item_emb_df_["image_emb"] = img_embs


Using device: cuda
Encoding 12100 items with CLIP...


CLIP encoding: 100%|██████████| 12100/12100 [40:40<00:00,  4.96it/s] 


In [15]:
# # Save to parquet file
# item_emb_df_.to_parquet(f'./{dataset_name}/item_emb.parquet', index=False)
#
# print("Item embeddings saved to item_emb.parquet.")

Item embeddings saved to item_emb.parquet.


In [16]:
# 转成 numpy 数组
text_emb = np.stack(item_emb_df['text_emb'].values)
image_emb = np.stack(item_emb_df['image_emb'].values)

# 分别保存
np.save(f'./{dataset_name}/item_text_emb.npy', text_emb)
np.save(f'./{dataset_name}/item_image_emb.npy', image_emb)

print(f"✅ Saved item_text_emb.npy and item_image_emb.npy to ./{dataset_name}/")
print("text_emb shape:", text_emb.shape)
print("image_emb shape:", image_emb.shape)

✅ Saved item_text_emb.npy and item_image_emb.npy to ./Beauty/
text_emb shape: (12101, 512)
image_emb shape: (12101, 512)


In [17]:
# pd.read_parquet(f'./{dataset_name}/item_emb.parquet')

,ItemID,text,image,price,brand,categories,text_emb,image_emb
0,1,title: WAWO 15 Color Professionl Makeup Eyesha...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,5.04,COKA,"[Beauty, Makeup, Face, Concealers & Neutralizers]","[-0.029019339010119438, 0.010487151332199574, ...","[-0.023478755727410316, 0.04270930588245392, 0..."
1,2,title: Xtreme Brite Brightening Gel 1oz.\nsale...,http://ecx.images-amazon.com/images/I/41QWW9v1...,19.99,Xtreme Brite,"[Beauty, Hair Care, Styling Products, Creams, ...","[0.0041499012149870396, 0.029814468696713448, ...","[0.016678830608725548, 0.014809356071054935, 0..."
2,3,title: Prada Candy By Prada Eau De Parfum Spra...,http://ecx.images-amazon.com/images/I/51iT2k6L...,65.86,Prada,"[Beauty, Fragrance, Women's, Eau de Parfum]","[-0.01337041612714529, -0.016733819618821144, ...","[-0.030022868886590004, -0.04415024071931839, ..."
3,4,title: Versace Bright Crystal Eau de Toilette ...,http://ecx.images-amazon.com/images/I/418LYGLE...,52.33,Versace,"[Beauty, Fragrance, Women's, Eau de Toilette]","[0.0030692084692418575, -0.0009710678132250905...","[-0.015382969751954079, -0.04214215278625488, ..."
4,5,title: Stella McCartney Stella\nsalesRank: {'B...,http://ecx.images-amazon.com/images/I/31L2n60J...,NaN,None,"[Beauty, Fragrance, Women's, Eau de Parfum]","[-0.01501120999455452, -0.0011238467413932085,...","[-0.04838697239756584, 0.021488163620233536, 0..."
...,...,...,...,...,...,...,...,...
12096,12096,"title: Moroccan Argan Oil - For Hair, Face, Sk...",http://ecx.images-amazon.com/images/I/41kTNm0k...,14.99,Natural Beauty,"[Beauty, Skin Care, Body, Moisturizers, Oils]","[-0.010838211514055729, 0.008691161870956421, ...","[-0.006330878473818302, 0.003624759614467621, ..."
12097,12098,title: LIME CRIME Velvetines - Wicked\nsalesRa...,http://ecx.images-amazon.com/images/I/41q7jpgt...,27.50,Lime Crime,"[Beauty, Makeup, Lips, Lipstick]","[0.0035878277849406004, 0.0315120592713356, -0...","[-0.04148492217063904, 0.011004919186234474, 0..."
12098,12099,title: Dr Song Rosehip Oil 4oz (4 oz)\nsalesRa...,http://ecx.images-amazon.com/images/I/412qdoPc...,19.99,None,"[Beauty, Skin Care, Face, Oils & Serums]","[-0.006727332714945078, 0.002433250891044736, ...","[0.007214454002678394, -0.020828960463404655, ..."
12099,12100,title: VITAMIN C SERUM 20% with Hyaluronic Aci...,http://ecx.images-amazon.com/images/I/31JTTyCU...,36.00,None,"[Beauty, Skin Care, Face, Creams & Moisturizer...","[-0.006677414756268263, -0.004898251499980688,...","[-0.04656478762626648, 0.01663064956665039, -0..."


In [18]:
# import numpy as np
# import pandas as pd
# from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
#
# # =============================================================
# # 复制，删除 text 和 image
# item_emb_df2 = item_emb_df_.copy()
# item_emb_df2.drop(columns=["text", "image"], inplace=True)
#
# # =============================================================
# # 1️⃣ 标准化价格
# df_std = item_emb_df2.copy()
# df_std["price"] = pd.to_numeric(df_std["price"], errors="coerce").fillna(0)
# mean_price = df_std["price"].mean()
# std_price = df_std["price"].std() if df_std["price"].std() != 0 else 1.0
# df_std["price_norm"] = ((df_std["price"] - mean_price) / std_price).astype(np.float32)
# df_std.drop(columns=["price"], inplace=True)
#
# # =============================================================
# # 2️⃣ 类别索引化：categories（多标签 → list[int]）
# mlb = MultiLabelBinarizer()
# cat_onehot = mlb.fit_transform(df_std["categories"])
#
# # 转成索引列表
# cat_idx_list = [np.where(row == 1)[0].astype(np.int64).tolist() for row in cat_onehot]
# df_std["categories_idx"] = cat_idx_list
# df_std.drop(columns=["categories"], inplace=True)
#
# # =============================================================
# # 3️⃣ 类别索引化：brand（单标签 → int）
# brand_series = (
#     df_std["brand"].copy()
#     .replace(["NaN", "None", None], np.nan)
#     .fillna("unknown")
# )
# brand_encoder = LabelEncoder()
# brand_idx = brand_encoder.fit_transform(brand_series)
# df_std["brand_idx"] = brand_idx.astype(np.int64)
# df_std.drop(columns=["brand"], inplace=True)
#
# # =============================================================
# # 4️⃣ 整理字段
# df_all = df_std.copy()
#
# # ---- price_norm: 独立列（float32）
# df_all["price_norm"] = df_all["price_norm"].apply(lambda x: np.array([x], dtype=np.float32))
#
# # ---- brand_idx: 独立列（int64）
# df_all["brand_idx"] = df_all["brand_idx"].apply(lambda x: np.array([x], dtype=np.int64))
#
# # ---- categories_idx: 已经是 list[int]
#
# # =============================================================
# print("✅ 数据处理完毕，可直接送入 nn.Linear / nn.Embedding")
# print("DataFrame columns:", df_all.columns.tolist())
# print(df_all[["price_norm", "brand_idx", "categories_idx"]].head())

✅ 数据处理完毕，可直接送入 nn.Linear / nn.Embedding
DataFrame columns: ['ItemID', 'text_emb', 'image_emb', 'price_norm', 'categories_idx', 'brand_idx']
     price_norm brand_idx      categories_idx
0  [-0.5683049]     [300]   [17, 55, 88, 144]
1  [0.19416447]    [1973]  [17, 63, 105, 212]
2    [2.533594]    [1456]   [17, 77, 99, 233]
3   [1.8435464]    [1896]   [17, 78, 99, 233]
4  [-0.8253514]    [2071]   [17, 77, 99, 233]


In [20]:
# # Save to parquet file
# df_all.to_parquet(f'./{dataset_name}/item_emb2.parquet', index=False)
#
# print("Item embeddings saved to item_emb2.parquet.")

Item embeddings saved to item_emb2.parquet.


In [19]:

# #.tolist()
#
# print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
# print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))
#
# # Save to parquet file
# item_emb_df.to_parquet(f'./{dataset_name}/item_emb.parquet', index=False)
#
# print("Item embeddings saved to item_emb.parquet.")
# embeddings = np.array([item['embedding'] for item in item_embeddings])
# np.save(f'./{dataset_name}/item_emb.npy', embeddings)

# print("Item embeddings saved to item_emb.npy.")


Item embeddings DataFrame shape: (12101, 6)
The first 3 rows of item embeddings DataFrame:
    ItemID                                               text  \
0       1  title: WAWO 15 Color Professionl Makeup Eyesha...   
1       2  title: Xtreme Brite Brightening Gel 1oz.\nsale...   
2       3  title: Prada Candy By Prada Eau De Parfum Spra...   

                                               image  price         brand  \
0  http://ecx.images-amazon.com/images/I/41Rn18Oe...   5.04          COKA   
1  http://ecx.images-amazon.com/images/I/41QWW9v1...  19.99  Xtreme Brite   
2  http://ecx.images-amazon.com/images/I/51iT2k6L...  65.86         Prada   

                                          categories  
0  [Beauty, Makeup, Face, Concealers & Neutralizers]  
1  [Beauty, Hair Care, Styling Products, Creams, ...  
2        [Beauty, Fragrance, Women's, Eau de Parfum]  
Item embeddings saved to item_emb.parquet.


In [51]:
# import pandas as pd
# data = pd.read_parquet('../data/Beauty/train.parquet')
# print("max history id:", max(max(x) for x in data['history']))
# print("max target id:", max(data['target']))

max history id: 12101
max target id: 12097


In [21]:
# import pandas as pd
# import numpy as np
#
# df = pd.read_parquet("../data/Beauty/item_emb.parquet")
# lengths = df["embedding"].apply(lambda x: len(x))
# print(lengths.value_counts().head())

KeyError: 'embedding'

In [1]:
# import pandas as pd
#
# df = pd.read_parquet("../data/Beauty/item_emb2.parquet")
#
# # 打印每个 embedding 的长度
# lens = df["text_emb"].apply(lambda x: len(x) if isinstance(x, list) else 0)
# print(lens.value_counts().head(10))

text_emb
0    12101
Name: count, dtype: int64
